# POC-7 Phase 3 — Analytical Track Development and Validation

**Approved analytical track:** Track A - Comparative Intelligence  
**Data version:** phase3-v2  
**Method version:** 1.0.0

This notebook documents and verifies the reproducible analytical workflow using the mandatory canonical dataset.

## 1. Analytical scope

Primary analytical question:

> How do lead value and days in stage vary across CRM funnel stages, acquisition sources and products within the available synthetic lead sample?

Decision supported:

Support descriptive operational review of observed differences across funnel stages, products and acquisition sources within the available synthetic sample.

In [ ]:
from pathlib import Path
import json
import pandas as pd

cwd = Path.cwd()

PROJECT_ROOT = None

for candidate in [cwd, *cwd.parents]:
    canonical_candidate = candidate / 'data' / 'canonical' / 'intelligence_data.csv'
    if canonical_candidate.exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        'Could not locate project root containing data/canonical/intelligence_data.csv'
    )

CANONICAL_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'canonical'
    / 'intelligence_data.csv'
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / 'data-science'
    / 'outputs'
)

print('Working directory:', cwd)
print('Project root:', PROJECT_ROOT)
print('Canonical path:', CANONICAL_PATH)
print('Output directory:', OUTPUT_DIR)


In [ ]:
df = pd.read_csv(CANONICAL_PATH)

print("Canonical rows:", len(df))
print("Canonical columns:", len(df.columns))

assert len(df) == 60
assert len(df.columns) == 20
assert df["data_version"].dropna().astype(str).unique().tolist() == ["phase3-v2"]

print("CANONICAL INPUT CHECK: PASS")


## 2. Analytical unit and approved dimensions

The analytical unit is one source CRM lead.

Approved comparison dimensions:
- funnel stage
- product category
- acquisition source

The available canonical structure contains 60 rows representing 30 source leads and their associated metric records.

In [ ]:
source_leads = df[df["record_type"] == "crm_lead"].copy()
metric_records = df[df["record_type"] == "crm_lead_metric"].copy()

assert len(source_leads) == 30
assert len(metric_records) == 30

lead_values = (
    source_leads
    .set_index("entity_id")["metric_value"]
    .astype(float)
)

days_by_entity = (
    metric_records
    .set_index("entity_id")["metric_value"]
    .astype(float)
)

analysis = source_leads[
    [
        "entity_id",
        "stage",
        "category",
        "subcategory",
        "metric_value",
    ]
].copy()

analysis = analysis.rename(
    columns={"metric_value": "lead_value"}
)

analysis["days_in_stage"] = (
    analysis["entity_id"]
    .map(days_by_entity)
)

assert len(analysis) == 30
assert analysis["lead_value"].notna().all()
assert analysis["days_in_stage"].notna().all()

print("Analytical rows:", len(analysis))
print("ANALYTICAL UNIT CHECK: PASS")


## 3. Baseline calculation

Baseline metrics provide the overall descriptive reference against which group-level results are interpreted.

In [ ]:
baseline = {
    "lead_count": int(len(analysis)),
    "total_lead_value": float(analysis["lead_value"].sum()),
    "average_lead_value": float(analysis["lead_value"].mean()),
    "average_days_in_stage": float(analysis["days_in_stage"].mean()),
    "median_days_in_stage": float(analysis["days_in_stage"].median()),
}

print("Baseline:")
for key, value in baseline.items():
    print(f"{key}: {value}")

assert baseline["lead_count"] == 30
assert baseline["total_lead_value"] == 2023000.0
assert round(baseline["average_lead_value"], 2) == 67433.33
assert round(baseline["average_days_in_stage"], 2) == 13.10
assert round(baseline["median_days_in_stage"], 2) == 10.00

print("BASELINE CHECK: PASS")


## 4. Comparative intelligence calculations

Track A ranks groups by total observed lead value in descending order. Ties are handled deterministically by group name.

In [ ]:
stage_summary = (
    analysis
    .groupby("stage", dropna=False)
    .agg(
        lead_count=("entity_id", "count"),
        total_lead_value=("lead_value", "sum"),
        average_lead_value=("lead_value", "mean"),
        average_days_in_stage=("days_in_stage", "mean"),
    )
    .reset_index()
    .sort_values(
        ["total_lead_value", "stage"],
        ascending=[False, True],
    )
)

product_summary = (
    analysis
    .groupby("category", dropna=False)
    .agg(
        lead_count=("entity_id", "count"),
        total_lead_value=("lead_value", "sum"),
        average_lead_value=("lead_value", "mean"),
        average_days_in_stage=("days_in_stage", "mean"),
    )
    .reset_index()
    .sort_values(
        ["total_lead_value", "category"],
        ascending=[False, True],
    )
)

source_summary = (
    analysis
    .groupby("subcategory", dropna=False)
    .agg(
        lead_count=("entity_id", "count"),
        total_lead_value=("lead_value", "sum"),
        average_lead_value=("lead_value", "mean"),
        average_days_in_stage=("days_in_stage", "mean"),
    )
    .reset_index()
    .sort_values(
        ["total_lead_value", "subcategory"],
        ascending=[False, True],
    )
)

print("Top stage:", stage_summary.iloc[0]["stage"])
print("Top product:", product_summary.iloc[0]["category"])
print("Top source:", source_summary.iloc[0]["subcategory"])

assert stage_summary.iloc[0]["stage"] == "Won"
assert stage_summary.iloc[0]["total_lead_value"] == 949000.0

assert product_summary.iloc[0]["category"] == "Payments"
assert product_summary.iloc[0]["total_lead_value"] == 942000.0

assert source_summary.iloc[0]["subcategory"] == "Website"
assert source_summary.iloc[0]["total_lead_value"] == 689000.0

print("COMPARATIVE RANKING CHECK: PASS")


## 5. Validation evidence

The notebook verifies the machine-generated validation artifacts from the reusable validation script rather than replacing them.

In [ ]:
validation_path = OUTPUT_DIR / "validation_metrics.json"
weak_case_path = OUTPUT_DIR / "weak_case_review.json"

with open(validation_path, "r", encoding="utf-8") as file:
    validation = json.load(file)

with open(weak_case_path, "r", encoding="utf-8") as file:
    weak_cases = json.load(file)

print("Validation status:", validation["validation_status"])
print("Calculation accuracy:", validation["calculation_accuracy"]["status"])
print(
    "Ranking stability:",
    all(
        item["common_order_stable"]
        for item in validation["small_group_sensitivity"].values()
    ),
)
print(
    "Weak cases:",
    len(weak_cases.get("weak_cases", [])),
)

assert validation["validation_status"] == "PASS"
assert validation["calculation_accuracy"]["status"] == "PASS"
assert validation["weak_case_count"] == 4
assert len(weak_cases.get("weak_cases", [])) == 4

print("VALIDATION EVIDENCE CHECK: PASS")


## 6. Intelligence output verification

In [ ]:
results_path = OUTPUT_DIR / "intelligence_results.json"
summary_path = OUTPUT_DIR / "intelligence_summary.json"

with open(results_path, "r", encoding="utf-8") as file:
    intelligence_results = json.load(file)

with open(summary_path, "r", encoding="utf-8") as file:
    intelligence_summary = json.load(file)

print("Result count:", intelligence_summary["result_count"])
print("Validation result:", intelligence_summary["validation_result"])
print("Weak-case count:", intelligence_summary["weak_case_count"])

assert intelligence_summary["data_version"] == "phase3-v2"
assert intelligence_summary["method_version"] == "1.0.0"
assert intelligence_summary["approved_track"] == (
    "Track A - Comparative Intelligence"
)
assert intelligence_summary["result_count"] == 16
assert intelligence_summary["validation_result"] == "PASS"
assert intelligence_summary["weak_case_count"] == 4

assert len(intelligence_results["results"]) == 16

print("INTELLIGENCE OUTPUT CHECK: PASS")


## 7. Interpretation and limitations

The approved interpretation is descriptive only. The analysis does not establish causation and is not predictive.

Important limitations include the synthetic dataset, the 30-lead analytical sample, sparse stage groups, lack of separate team/location fields, and the absence of a full longitudinal time series.

In [ ]:
print("=" * 60)
print("TRACK A NOTEBOOK VALIDATION SUMMARY")
print("=" * 60)
print("Data version: phase3-v2")
print("Method version: 1.0.0")
print("Approved track: Track A - Comparative Intelligence")
print("Analytical rows: 30")
print("Baseline total lead value: 2023000.0")
print("Top stage: Won")
print("Top product: Payments")
print("Top source: Website")
print("Validation status: PASS")
print("Weak cases reviewed: 4")
print("Interpretation: DESCRIPTIVE ONLY")
print("Predictive use: NOT SUPPORTED")
print("Causal interpretation: NOT SUPPORTED")
print("=" * 60)
print("TRACK A NOTEBOOK CHECK: PASS")
